# ???????????? ??????? ?3 ?? ???? "????? ??????-??????????? ? DS-??????".

1. ??????? ????????? ?????? ??????? ?? ?????? ?????? ??? (train_case2.csv): ??????, ???????, ??? ? ?.? - ?? ??? ????? 2-3 ????????
2. ??? ???????? ??????? ??????????? ???????????? ??????????????
3. ??????? ????????? ?????????? ??????? ?? ???????? ???????? ?????????????: pr/rec/auc/f_score (????? ? ???? ???????, ??? ?????? - ??????, ? ??????? - ???????)
4. ??????? ?????? ? ???, ????? ?????? ?????????? ? ??????? ????? ??????
5. (???????????? ??????) ????? ??????? (precision_recall_curve ??? roc_auc_curve) ?????? ???????? ? ?????? ???????? ?????????? ???????? (????? ???????? ?????? ?? ??????? ??????? ?????? ??? ???????). 

p.s.? ??????? ????? ???????????, ???? ????????? ??? ?? ??????? roc auc curve ? ??????????? ????? ??????:

??????? 100000 ????????, ?? ??????? ?????? 100 - ????? "1" (99900 - ????? "0", ??????????????). 
????????, ? ??? ??? ??????:

- ?????? ???????? 100 ???????? ??? ????? 1, ?? TP = 90
- ?????? ???????? 1000 ???????? ??? ????? 1, ?? TP ????? ?? - 90

????? ?????? ????? ? ??????? ? ??? ????????? ????? ??????? ????? - roc_auc_curve ??? precision_recall_curve?

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import xgboost
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_recall_curve, roc_auc_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
df = pd.read_csv('train_case2.csv', ';')
df.sample(10, random_state=12)

In [ ]:
#???????? ?????? ?? train/test
X_train, X_test, y_train, y_test = train_test_split(df.drop('cardio', 1), df['cardio'], random_state=0)

In [ ]:
# ????? ??????? (????????)
class ColumnSelector(BaseEstimator, TransformerMixin):
    """
    Transformer to select a single column from the data frame to perform additional transformations on
    """
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.key]

# ??????? ??????? ? ????????? ???????
class NumberSelector(BaseEstimator, TransformerMixin):
    """
    Transformer to select a single column from the data frame to perform additional transformations on
    Use on numeric columns in the data
    """
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[[self.key]]

# ???????????? ?????????????? ???????????? ?????????
class OHEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key
        self.columns = []

    def fit(self, X, y=None):
        self.columns = [col for col in pd.get_dummies(X, prefix=self.key).columns]
        return self

    def transform(self, X):
        X = pd.get_dummies(X, prefix=self.key)
        test_columns = [col for col in X.columns]
        for col_ in test_columns:
            if col_ not in self.columns:
                X[col_] = 0
        return X[self.columns]


from sklearn.preprocessing import StandardScaler

# ???????? ????????, ???????????? ? ???????? ????????
continuos_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_cols = ['gender', 'cholesterol']
base_cols = ['gluc', 'smoke', 'alco', 'active']

# ?????? ?????????? ??? ??????? ????????
continuos_transformers = []
cat_transformers = []
base_transformers = []

for cont_col in continuos_cols:
    transfomer =  Pipeline([
                ('selector', NumberSelector(key=cont_col)),
                ('standard', StandardScaler())
            ])
    continuos_transformers.append((cont_col, transfomer))
    
for cat_col in cat_cols:
    cat_transformer = Pipeline([
                ('selector', ColumnSelector(key=cat_col)),
                ('ohe', OHEEncoder(key=cat_col))
            ])
    cat_transformers.append((cat_col, cat_transformer))
    
for base_col in base_cols:
    base_transformer = Pipeline([
                ('selector', NumberSelector(key=base_col))
            ])
    base_transformers.append((base_col, base_transformer))

In [ ]:
feats = FeatureUnion(continuos_transformers+cat_transformers+base_transformers)
feature_processing = Pipeline([('feats', feats)])

feature_processing.fit_transform(X_train)

In [ ]:
result_dict = {'LogReg':[], 'XGB':[], 'kNN':[]}

## ??????? ?1, 2

#### 1. Logistic Regression

In [ ]:
%%time
classifier = Pipeline([
    ('features',feats),
    ('classifier', LogisticRegression(random_state = 42)),
])

cv_scores = cross_val_score(classifier, X_train, y_train, cv=16, scoring='roc_auc')
cv_score = np.mean(cv_scores)
cv_score_std = np.std(cv_scores)

classifier.fit(X_train, y_train)
y_score = classifier.predict_proba(X_test)[:, 1]

In [ ]:
print('CV score is {}+-{}'.format(cv_score, cv_score_std))

In [ ]:
b=1
precision, recall, thresholds = precision_recall_curve(y_test.values, y_score)
fscore = (1+b**2)*(precision * recall) / (b**2*precision + recall)
ix = np.argmax(fscore)
ras = roc_auc_score(y_test, y_score)

result_dict['LogReg'].append(fscore[ix])
result_dict['LogReg'].append(precision[ix])
result_dict['LogReg'].append(recall[ix])
result_dict['LogReg'].append(ras)

print('Best Threshold=%f, F-Score=%.3f, Precision=%.3f, Recall=%.3f' % (thresholds[ix], 
                                                                        fscore[ix],
                                                                        precision[ix],
                                                                        recall[ix]))

#### 2. Xgboost

In [ ]:
%%time
classifier = Pipeline([
    ('features',feats),
    ('classifier', xgboost.XGBClassifier(n_estimators=100, random_state=0)),
])

cv_scores = cross_val_score(classifier, X_train, y_train, cv=5, scoring='roc_auc')
cv_score = np.mean(cv_scores)
cv_score_std = np.std(cv_scores)

classifier.fit(X_train, y_train)
y_score = classifier.predict_proba(X_test)[:, 1]

In [ ]:
print('CV score is {}+-{}'.format(cv_score, cv_score_std))

In [ ]:
b=1
precision, recall, thresholds = precision_recall_curve(y_test.values, y_score)
fscore = (1+b**2)*(precision * recall) / (b**2*precision + recall)
ix = np.argmax(fscore)
ras = roc_auc_score(y_test, y_score)

result_dict['XGB'].append(fscore[ix])
result_dict['XGB'].append(precision[ix])
result_dict['XGB'].append(recall[ix])
result_dict['XGB'].append(ras)

print('Best Threshold=%f, F-Score=%.3f, Precision=%.3f, Recall=%.3f' % (thresholds[ix], 
                                                                        fscore[ix],
                                                                        precision[ix],
                                                                        recall[ix]))

#### 3. k-NN

In [ ]:
%%time
classifier = Pipeline([
    ('features',feats),
    ('classifier', KNeighborsClassifier(n_neighbors = 5))
])

cv_scores = cross_val_score(classifier, X_train, y_train, cv=5, scoring='roc_auc')
cv_score = np.mean(cv_scores)
cv_score_std = np.std(cv_scores)

classifier.fit(X_train, y_train)
y_score = classifier.predict_proba(X_test)[:, 1]

In [ ]:
print('CV score is {}+-{}'.format(cv_score, cv_score_std))

In [ ]:
b=1
precision, recall, thresholds = precision_recall_curve(y_test.values, y_score)
fscore = (1+b**2)*(precision * recall) / (b**2*precision + recall)
ix = np.argmax(fscore)
ras = roc_auc_score(y_test, y_score)

result_dict['kNN'].append(fscore[ix])
result_dict['kNN'].append(precision[ix])
result_dict['kNN'].append(recall[ix])
result_dict['kNN'].append(ras)

print('Best Threshold=%f, F-Score=%.3f, Precision=%.3f, Recall=%.3f' % (thresholds[ix], 
                                                                        fscore[ix],
                                                                        precision[ix],
                                                                        recall[ix]))

In [ ]:
result_dict

## ??????? ?3

In [ ]:
result_df = pd.DataFrame(result_dict, index=['F-score', 'Precision', 'Recall', 'Roc Auc Score'])
result_df = result_df.T
result_df

## ??????? ?4

?? ?????????? ??????????? ????? ??????? ?????, ??? ?????? ?????????? ?? ???? ???????? ? ????????? **k-NN**. ????????? **Logistic Regression** ? **XGboost** ???????? ???????? ?????????? ??????????, ?? **XGboost** ???-???? ??????? ?????.

??????, ????? ???????? ???????? ?? ??, ??? ?? ??????? ????????? **XGboost** ?????? ???????? **Logistic Regression**.

## ??????? ?5

? ?????? ????????????? ????????? ??? ????????, ??? ??????? *roc_auc_curve* ?????????? ?? ???????? ??? ?????????, ? ??????? ???????????? ??????? ????????? ???????. ? ????? ??????? ????? ????? ???????????? ??????? **F-score** ? ?????????????? *precision_recall_curve*.

? ??????? ???????????? ??????????? ??????:

??????? 100000 ????????, ?? ??????? ?????? 100 - ????? "1" (99900 - ????? "0", ??????????????). 
????????, ? ??? ??? ??????:

- ?????? ???????? 100 ???????? ??? ????? 1, ?? TP = 90
- ?????? ???????? 1000 ???????? ??? ????? 1, ?? TP ????? ?? - 90

F-score:

$$precision = \frac{TP}{TP+FP}$$

$$recall = \frac{TP}{TP+FN}$$

$$Fscore = (1+\beta^2)\cdot\frac{precision \cdot recall}{(\beta^2 \cdot precision)+recall}$$

Roc Auc score:

$$TPR = \frac{TP}{TP+FN}$$

$$FPR = \frac{FP}{FP+TN}$$

?????? ??????

In [ ]:
y_true = [1]*100 + [0]*99900
y_score = [1]*90 + [0]*10 + [1]*10 + [0]*99890

In [ ]:
cnf_matrix = confusion_matrix(y_true, y_score)
TN = cnf_matrix[0][0]
FN = cnf_matrix[1][0]
TP = cnf_matrix[1][1]
FP = cnf_matrix[0][1]
precision = TP/(TP+FP)
recall = TP/(TP+FN)
beta = 1
f_score = 2*precision*recall/(precision + recall)
ras = roc_auc_score(y_true, y_score)
print('F-Score=%.3f, \nPrecision=%.3f, \nRecall=%.3f, \nRoc Auc Score=%.3f' % (f_score, precision, recall, ras))

?????? ??????

In [ ]:
y_true = [1]*100 + [0]*99900
y_score = [1]*90 + [0]*10 + [1]*910 + [0]*98990

In [ ]:
cnf_matrix = confusion_matrix(y_true, y_score)
TN = cnf_matrix[0][0]
FN = cnf_matrix[1][0]
TP = cnf_matrix[1][1]
FP = cnf_matrix[0][1]
precision = TP/(TP+FP)
recall = TP/(TP+FN)
beta = 1
f_score = 2*precision*recall/(precision + recall)
ras = roc_auc_score(y_true, y_score)
print('F-Score=%.3f, \nPrecision=%.3f, \nRecall=%.3f, \nRoc Auc Score=%.3f' % (f_score, precision, recall, ras))

**?????:**

?????????? ?????????? ???? ????????. ? ?????? ?????? ??? ??????? ? ???????????????????? ??????? ?? ??? ?????? ???????, ?????? ??? ?????? ?????????? ????????? ?????? ???????? ? ??? ?? ????, ??? ? ??????? ?????? ? ???????. ??????, ?? ?????? ??????, ?? ????? ??? ?????? ??????????? ????? ????? FP ???????, ??? ??????? ? ???????? ???????? ???????? ? ?????????????? F-score. 

???? ?????? ?????? ?? ???????? ??? ???????????????????? ???????, ?? ??????????? ???????????????? ???????????? F-score, ??????? ?????????? FP ?????? ? ??????? ????????????? ??? ? ???, ??? ? ??????? ???-?? ?? ???. Roc Auc Score ?????? ??????????? ????? ?? ???????????? ?? ????????? ?????? ? ???????, ???? ?? ????? ???????????? ???, ?? ?? ????? ? ?? ??????, ??? ? ??? ????????.

??????????? ?????? ?????? ??????? ????????????, ????????? ?????????? F-score ? ??? ????.

---